# 02 · Regional analysis: where is the shift fastest or slowest, and why?

**Business question:** How fast is Sweden shifting to battery-electric cars among private buyers, where is it fastest or slowest and why, and what BEV share is plausible by 2028?

This notebook covers **where and why**. It compares Sweden's 21 counties (län) on the private BEV share, looks at how each county moved through the bonus surge, the post-bonus fall and the 2026 restart, and tests which county characteristics go with higher shares.

Notebook 01 established the national pattern: private BEV share rose from 21% (2021) to 39% (2022), fell to about 32% in 2024–25, and is 38.6% in Jan–Aug 2026.

## 1. Setup and data

**Choices made here**
- **Private buyers only** (see `CLAUDE.md`). Company and leasing cars are registered in the lessor's county: Stockholm has **about half of all company registrations but under a quarter of private ones** (printed below). Dealer registrations are demo cars. Neither shows where households buy.
- **Periods:** full calendar years 2021–2025, plus **Jan–Aug 2025 and Jan–Aug 2026**, so that 2026 is compared with the same months a year earlier (the seasonality shown in 01).
- **County `99` (Unknown)** is excluded. It has essentially no private registrations (company and dealer cars only).
- **Ratios of sums** throughout: county-period share = BEV registrations ÷ all registrations in that county and period.
- Private-buyer data starts in 2021, so this notebook can't go further back. The 2010–2020 county series covers all owners and would mix in the leasing distortion.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy import stats

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))
import viz
from viz import BLUE, ORANGE, AQUA, RED, INK, INK2, MUTED, AXIS, GREY_DARK, GREY_LIGHT, SURFACE, PCT
viz.apply_style()
PROC, FIGS = ROOT / "data" / "processed", ROOT / "docs" / "figures"

fact = pd.read_csv(PROC / "fact_registrations_county_monthly.csv", parse_dates=["month"], dtype={"county_code": str})
panel = pd.read_csv(PROC / "county_year_panel.csv", dtype={"county_code": str})

priv = fact[(fact.owner_type == "private") & (fact.county_code != "99")].copy()
LAST = priv.month.max()
priv["year"] = priv.month.dt.year
comp = fact[fact.owner_type == "company"]
print(f"Stockholm's share of company registrations: {comp[comp.county_code == '01'].registrations.sum() / comp.registrations.sum():.1%}"
      f" vs private: {priv[priv.county_code == '01'].registrations.sum() / priv.registrations.sum():.1%}")
print(f"Unknown-county share of private registrations: "
      f"{fact[(fact.owner_type == 'private') & (fact.county_code == '99')].registrations.sum() / fact[fact.owner_type == 'private'].registrations.sum():.3%}")

def county_period(df):
    t = df.pivot_table(index=["county_name", "year"], columns="fuel_type", values="registrations",
                       aggfunc="sum", fill_value=0)
    t["total"] = t.sum(axis=1)
    return t

cy = county_period(priv[priv.year < LAST.year])                     # full years 2021-2025
ytd = county_period(priv[priv.month.dt.month <= LAST.month])        # Jan-Aug every year
YTD_PREV, YTD_CUR = f"Jan–{LAST:%b} {LAST.year - 1}", f"Jan–{LAST:%b} {LAST.year}"

def shares(fuel):
    s = (cy[fuel] / cy.total).unstack()
    s.columns = s.columns.astype(str)
    y = (ytd[fuel] / ytd.total).unstack()
    s[YTD_PREV], s[YTD_CUR] = y[LAST.year - 1], y[LAST.year]
    return s

bev, phev = shares("BEV"), shares("PHEV")
vol = cy.total.unstack()
print(f"{bev.shape[0]} counties; private registrations in 2025 range from "
      f"{vol[2025].min():,.0f} ({vol[2025].idxmin()}) to {vol[2025].max():,.0f} ({vol[2025].idxmax()})")

Stockholm's share of company registrations: 50.8% vs private: 23.1%
Unknown-county share of private registrations: 0.000%
21 counties; private registrations in 2025 range from 356 (Gotland) to 23,211 (Stockholm)
